# 07 · Multilingual text models — M6 classifier + M7 embedder

> **SYNTHETIC DATA workflow.** Trains on LLM-written and template complaints (never real citizen reports) and evaluates on held-out request groups, the original template eval set and — if present — the team's own **gold** test set.

**M6** = `intfloat/multilingual-e5-small` fine-tuned as the 27-way civic classifier (replaces the weak B0 fallback). **M7** = the same *pretrained* encoder exported as a sentence embedder (cross-language duplicate detection). Both are exported to **ONNX** (+ int8 when it agrees with fp32) and run on CPU in the backend.

**Before this notebook** (on your laptop, with your `.env`):
`python -m ai.training.src.text_corpus.generate --out D:/civic_corpus/shards` → `python -m ai.training.src.text_corpus.build --shards D:/civic_corpus/shards --out D:/civic_corpus/corpus --templates ai/artifacts/datasets/synthetic_v1 [--gold gold.csv]`.
Upload the `corpus` folder as a private Kaggle Dataset (or run this notebook locally with `CIVIC_CORPUS_DIR=D:/civic_corpus/corpus`).

**Settings:** *Accelerator* **GPU T4 ×2** (both GPUs are used through `DataParallel`; the effective batch and the loss are the single-GPU ones; `CIVIC_NUM_GPUS=1` forces one GPU), a single GPU/P100 or a local RTX 4050 also work; Internet On (clones the repo, downloads the encoder). `SMOKE = True` runs a 2-epoch proof first.

**Train the stronger base encoder instead:** change only the settings cell (below) — uncomment its three `os.environ[...]` lines (`CIVIC_TEXT_MODEL`, `CIVIC_TEXT_PREFIX`, `CIVIC_TEXT_VERSION`). M7 is the pretrained encoder exported as it is (not fine-tuned), so only M6 is trained.

In [ ]:
# --- 0. Locate the CivicConnect repo and set up a work directory (works on Kaggle and locally) ---------------
import importlib.util
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

def find_repo() -> Path:
    """Return a writable directory that contains ai/inference/schemas.py."""
    marker = Path("ai/inference/schemas.py")
    if os.environ.get("CIVIC_REPO"):                                   # explicit override
        return Path(os.environ["CIVIC_REPO"])
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:                                    # running from inside the repo
        if (p / marker).exists():
            return p
    if ON_KAGGLE:                                                      # attached Kaggle Dataset containing the repo or just ai/
        for hit in sorted(Path("/kaggle/input").glob("**/ai/inference/schemas.py"))[:1]:
            src = hit.parents[2]                                       # .../ai/inference/schemas.py -> parent of ai/
            dst = Path("/kaggle/working/repo")
            if not dst.exists():
                shutil.copytree(src, dst)                              # /kaggle/input is read-only
            return dst
        url = os.environ.get("CIVIC_GIT_URL", "https://github.com/tanujb03/CivicConnect_v2")                          # default: clone the project repo's `tanuj` branch (needs Internet ON); override with CIVIC_GIT_URL / CIVIC_GIT_REF
        if url:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", os.environ.get("CIVIC_GIT_REF", "tanuj"), url, "/kaggle/working/repo"], check=True)
            return Path("/kaggle/working/repo")
    raise SystemExit("Cannot find the repo. On Kaggle: turn Internet ON (the notebook clones the project from GitHub) or add a Dataset containing the repo's `ai/` folder (see ai/training/README.md).")

REPO = find_repo()
sys.path.insert(0, str(REPO))
WORK = Path(os.environ.get("CIVIC_WORKDIR") or ("/kaggle/working" if ON_KAGGLE else tempfile.mkdtemp(prefix="civic_work_")))
WORK.mkdir(parents=True, exist_ok=True)

_MODULE = {"scikit-learn": "sklearn", "onnxruntime": "onnxruntime", "onnx": "onnx", "tokenizers": "tokenizers", "sentencepiece": "sentencepiece"}

def ensure(*pkgs):
    """pip install only what is missing (Kaggle images already ship numpy/scipy/scikit-learn/pydantic/httpx)."""
    missing = [p for p in pkgs if importlib.util.find_spec(_MODULE.get(p, p)) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

ensure("numpy", "scipy", "scikit-learn", "pydantic", "httpx")
print(f"repo : {REPO}\nwork : {WORK}\npython {sys.version.split()[0]} | kaggle={ON_KAGGLE}")

## 1 · Settings

In [ ]:
# --- To train the BASE model (stronger, ~2.3x slower) change ONLY these three lines: uncomment them, run the cell, run the rest. Use a new VERSION so the artifacts do not overwrite the small ones. ---
# os.environ["CIVIC_TEXT_MODEL"] = "intfloat/multilingual-e5-base"
# os.environ["CIVIC_TEXT_PREFIX"] = "query: "          # e5 models expect "query: "
# os.environ["CIVIC_TEXT_VERSION"] = "2-base"

SMOKE = True            # True: 2 epochs on a subset (~5 min). Set False for the real run.
MODEL_NAME = os.environ.get("CIVIC_TEXT_MODEL", "intfloat/multilingual-e5-small")   # "intfloat/multilingual-e5-base" is stronger and ~2.3x slower (use BATCH=16 on a 6 GB GPU)
EPOCHS, BATCH, MAX_LEN, LR = 6, 32, 64, 3e-5      # BATCH = EFFECTIVE batch per optimizer step, split over the GPUs (T4 x2: 16 each)
PREFIX = os.environ.get("CIVIC_TEXT_PREFIX", "query: ")      # e5 models expect "query: "; use "" for models that do not
SEED = 42
GRAD_ACCUM = int(os.environ.get("CIVIC_GRAD_ACCUM", "1"))   # micro-batches per optimizer step (same effective batch, 1/GRAD_ACCUM of the memory; raise it if the base model runs out of GPU memory)
VERSION = os.environ.get("CIVIC_TEXT_VERSION", "1")
if SMOKE:
    EPOCHS = 2

## 2 · Find the corpus and install what is missing

In [ ]:
import json

ensure("transformers", "tokenizers", "onnxruntime", "onnx", "sentencepiece")

CORPUS = Path(os.environ["CIVIC_CORPUS_DIR"]) if os.environ.get("CIVIC_CORPUS_DIR") else next(
    (m.parent for m in sorted(Path("/kaggle/input").glob("**/manifest.json")) if "civic_text_corpus" in m.read_text(encoding="utf-8")[:300]), None)
assert CORPUS and (CORPUS / "train.jsonl").exists(), "attach the corpus Dataset (train/val/test.jsonl + manifest.json) or set CIVIC_CORPUS_DIR"
MF = json.loads((CORPUS / "manifest.json").read_text(encoding="utf-8"))
print("corpus:", CORPUS, "|", {k: v["rows"] for k, v in MF["splits"].items()}, "| gold:", (MF.get("gold") or {}).get("rows", 0))
print("languages in train:", MF["splits"]["train"]["by_language"], "| sources:", MF["splits"]["train"]["by_source"])
from ai.training.src import hardware as hw  # noqa: E402

GPUS = hw.plan_gpus()                                      # nvidia-smi + CUDA_VISIBLE_DEVICES; CIVIC_NUM_GPUS=1 forces a single GPU, =0 CPU
NUM_GPUS = GPUS.used if GPUS.detected else None            # None: no nvidia-smi, let torch decide
print(hw.banner(GPUS, hw.split_batch(BATCH, GPUS.used, must_divide=False)) + ("" if GPUS.used else "   (CPU: fine for the smoke run, slow for the real one)"))

## 3 · Fine-tune M6 and evaluate on every held-out set

In [ ]:
from ai.training.src.text_model import export as ex
from ai.training.src.text_model import train as tr

if SMOKE:                                                    # tiny subset so every cell can be proven quickly
    for name in ("train", "val", "test"):
        rows = tr.read_jsonl(CORPUS / f"{name}.jsonl")
        sub = WORK / "corpus_smoke"
        sub.mkdir(exist_ok=True)
        (sub / f"{name}.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in rows[:: max(1, len(rows) // 1500)]) + "\n", encoding="utf-8")
    if (CORPUS / "gold.jsonl").exists():
        (WORK / "corpus_smoke" / "gold.jsonl").write_text((CORPUS / "gold.jsonl").read_text(encoding="utf-8"), encoding="utf-8")
    TRAIN_DIR = WORK / "corpus_smoke"
else:
    TRAIN_DIR = CORPUS
if GPUS.used:
    hw.start_gpu_watch(GPUS.used, out_file=WORK / "gpu_watch.txt")     # prints per-GPU utilisation 2 and 15 minutes into training (a second cell cannot run while this one is busy)
model, tok, info = tr.train(TRAIN_DIR, MODEL_NAME, epochs=EPOCHS, batch=BATCH, lr=LR, max_len=MAX_LEN, seed=SEED, prefix=PREFIX, num_gpus=NUM_GPUS, grad_accum=GRAD_ACCUM,
                            template_eval=REPO / "ai/evaluation/datasets/intake_eval.v1.jsonl")

## 4 · Compare with the old B0 classifier
B0 (char n-grams on short template lines) scored **category 0.616 / subcategory 0.546** on the *template held-out families* (`template_unseen_families` below is the same file). The LLM-group test and the **gold** set are new, harder and more realistic: judge M6 mainly on those.

In [ ]:
for name, r in info["results"].items():
    print(f"{name:26s} n={r['n']:5d}  subcategory {r['subcategory_accuracy']:.3f}  category {r['category_accuracy']:.3f}  macro-F1 {r['macro_f1']:.3f}   by language: " +
          ", ".join(f"{k} {v['subcategory_accuracy']:.2f}(n={v['n']})" for k, v in r["by_language"].items()))
print("\ntemperature (calibration on val):", info["temperature"])
print("training hardware:", {k: info["hardware"][k] for k in ("gpu_count", "gpu_names", "parallel", "batch_effective", "batch_per_gpu", "grad_accum", "wall_seconds")}, "| seconds per epoch:", info["hardware"]["epoch_seconds"])
if "gold" not in info["results"]:
    print("\nNO GOLD SET in this corpus: the numbers above only measure agreement with LLM-written text. Add your own gold.csv (see text_corpus.build --gold-template) for a real check.")

## 5 · Export M6 (classifier) and M7 (embedder) to ONNX and verify the backend runtime on the exported files

In [ ]:
test_rows = tr.read_jsonl(TRAIN_DIR / "test.jsonl")
sample = [r["text"] for r in test_rows[:60]]
ART = WORK / "artifacts"
M6 = ex.export_classifier(model, tok, info, ART / "civic_text_m6" / VERSION, version=VERSION, sample_texts=sample, quantize=True,
                          extra={"corpus_manifest": {k: MF[k] for k in ("splits", "seed", "leakage_rule")}, "smoke_run": SMOKE})
M7 = ex.export_embedder(MODEL_NAME, tok, ART / "civic_embed_m7" / VERSION, version=VERSION, prefix=PREFIX, max_len=MAX_LEN * 2, sample_texts=sample[:20])
from ai.training.src.text_model.runtime import load_backend_text_model  # noqa: E402

TM = load_backend_text_model(REPO)
clf, emb = TM.OnnxTextClassifier(M6), TM.OnnxEmbedder(M7)
rows = test_rows[:300]
hit = sum(clf.predict(r["text"]).label_id == r["label_id"] for r in rows)
print(f"backend ONNX classifier on {len(rows)} test rows: subcategory accuracy {hit / len(rows):.3f}  (model file: {clf.model_file})")
print("classifier ONNX/torch parity:", json.loads((M6 / "manifest.json").read_text())["torch_onnx_parity"], "| int8:", json.loads((M6 / "manifest.json").read_text())["int8"])
a, b, c = emb.embed(["there is a big pothole near the school", "स्कूल के पास सड़क पर बड़ा गड्ढा है", "the street light is not working"])
cos = lambda x, y: sum(p * q for p, q in zip(x, y))  # noqa: E731
print(f"embedder cross-language check: en~hi (same issue) {cos(a, b):.3f}   en~other issue {cos(a, c):.3f}   (dimension {emb.dimension})")

## 6 · Bundle

In [ ]:
import hashlib
import zipfile

BUNDLE = WORK / f"civic_text_models_{'smoke_' if SMOKE else ''}{VERSION}.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for d in (M6, M7):
        for f in sorted(d.iterdir()):
            z.write(f, f"{d.parent.name}/{d.name}/{f.name}")
print("bundle:", BUNDLE, f"({BUNDLE.stat().st_size / 1e6:.1f} MB)\nsha256:", hashlib.sha256(BUNDLE.read_bytes()).hexdigest())
print("\nHand-off: unzip into ai/artifacts/ and set in .env:\n  AI_TEXT_ONNX_PATH=ai/artifacts/civic_text_m6/" + VERSION + "\n  AI_EMBED_ONNX_PATH=ai/artifacts/civic_embed_m7/" + VERSION)